# XDMA + CDMA + DCIM BRAM Test

This notebook tests the `xdma_dcim_bram` design through the host-visible address map:

- XDMA writes raw int8 operands into `global_bram`.
- AXI CDMA copies weight/activation data from `global_bram` into PE `ibuf`.
- AXI GPIO configures and starts the PE/DCIM int8 tile engine.
- AXI CDMA copies PE `obuf` results back into `global_bram`.
- XDMA reads the result and compares it against a NumPy golden model.


In [1]:
from pathlib import Path
import hashlib
import subprocess
import time
from dataclasses import dataclass

import numpy as np

def find_tests_dir(cwd: Path) -> Path:
    for base in (cwd, *cwd.parents):
        if base.name == "tests":
            return base
        if (base / "tests").exists():
            return base / "tests"
    return cwd


cwd = Path.cwd()
TESTS_DIR = find_tests_dir(cwd)
TEST_DIR = TESTS_DIR / "xdma_dcim_bram" if (TESTS_DIR / "xdma_dcim_bram").exists() else cwd

xdma_candidates = [
    TEST_DIR / "bin" / "xdma_rw.exe",
    TESTS_DIR / "bin" / "xdma_rw.exe",
    TESTS_DIR / "xdma_bram" / "bin" / "xdma_rw.exe",
]
XDMA_EXE = next((path for path in xdma_candidates if path.exists()), None)
assert XDMA_EXE is not None, "Cannot find xdma_rw.exe; expected it in tests/bin or tests/xdma_bram/bin"

xdma_info_candidates = [
    TEST_DIR / "bin" / "xdma_info.exe",
    TESTS_DIR / "bin" / "xdma_info.exe",
    TESTS_DIR / "xdma_bram" / "bin" / "xdma_info.exe",
    XDMA_EXE.with_name("xdma_info.exe"),
]
XDMA_INFO_EXE = next((path for path in xdma_info_candidates if path.exists()), None)

DATA_DIR = TEST_DIR / "data"
READBACK_DIR = TEST_DIR / "readbacks"
REG_DIR = TEST_DIR / "reg_io"
for path in (DATA_DIR, READBACK_DIR, REG_DIR):
    path.mkdir(parents=True, exist_ok=True)

CHANNEL = 0
CMD_TIMEOUT_SEC = 30

GLOBAL_BASE = 0x10000000
IBUF_BASE = 0x10010000
OBUF_BASE = 0x10020000
CDMA_BASE = 0x10030000
PE_CFG0_BASE = 0x10040000  # wsrc_base_addr in ibuf
PE_CFG1_BASE = 0x10041000  # asrc_base_addr in ibuf
PE_CFG2_BASE = 0x10042000  # dst_addr in obuf
PE_CFG3_BASE = 0x10043000  # raw_rows
PE_CTRL_BASE = 0x10044000
PE_STATUS_BASE = 0x10045000

WINDOW_BYTES = 64 * 1024
BRAM_WORD_BYTES = 32
MODE_INT8 = 0b110

CDMA_CR = 0x00
CDMA_SR = 0x04
CDMA_SA = 0x18
CDMA_DA = 0x20
CDMA_BTT = 0x28
CDMA_SR_IDLE = 1 << 1
CDMA_SR_ERR_MASK = (1 << 4) | (1 << 5) | (1 << 6) | (1 << 8) | (1 << 9) | (1 << 10)

print(f"XDMA tool: {XDMA_EXE}")
print(f"XDMA info: {XDMA_INFO_EXE if XDMA_INFO_EXE else 'not found'}")
print(f"Test dir : {TEST_DIR}")
print(f"global_bram: 0x{GLOBAL_BASE:08X}..0x{GLOBAL_BASE + WINDOW_BYTES:08X}")
print(f"PE ibuf    : 0x{IBUF_BASE:08X}..0x{IBUF_BASE + WINDOW_BYTES:08X}")
print(f"PE obuf    : 0x{OBUF_BASE:08X}..0x{OBUF_BASE + WINDOW_BYTES:08X}")

XDMA tool: /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/bin/xdma_rw.exe
XDMA info: /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/bin/xdma_info.exe
Test dir : /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/xdma_dcim_bram
global_bram: 0x10000000..0x10010000
PE ibuf    : 0x10010000..0x10020000
PE obuf    : 0x10020000..0x10030000


In [2]:
@dataclass
class CmdResult:
    cmd: list[str]
    returncode: int
    stdout: str
    stderr: str
    elapsed_sec: float

    @property
    def ok(self) -> bool:
        return self.returncode == 0


def _subprocess_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def run_cmd(cmd: list[str], timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    start = time.perf_counter()
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
        return CmdResult(cmd, cp.returncode, cp.stdout, cp.stderr, time.perf_counter() - start)
    except subprocess.TimeoutExpired as exc:
        elapsed = time.perf_counter() - start
        stderr = _subprocess_text(exc.stderr)
        if stderr:
            stderr += "\n"
        stderr += f"TIMEOUT: command did not finish within {timeout}s"
        return CmdResult(cmd, -124, _subprocess_text(exc.stdout), stderr, elapsed)


def print_result(name: str, result: CmdResult) -> None:
    print(f"{name}: {'PASS' if result.ok else 'FAIL'}, ret={result.returncode}, elapsed={result.elapsed_sec:.3f}s")
    print("CMD:", " ".join(result.cmd))
    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())


def xdma_write(addr: int, payload_path: Path, timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    return run_cmd([str(XDMA_EXE), f"h2c_{CHANNEL}", "write", hex(addr), "-b", "-f", str(payload_path)], timeout=timeout)


def xdma_read(addr: int, size: int, out_path: Path, timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    if out_path.exists():
        out_path.unlink()
    return run_cmd([str(XDMA_EXE), f"c2h_{CHANNEL}", "read", hex(addr), "-l", hex(size), "-b", "-f", str(out_path)], timeout=timeout)


def mmio_write_u32(addr: int, value: int, label: str = "") -> CmdResult:
    name = label or f"wr_{addr:08X}"
    path = REG_DIR / f"{name}.bin"
    path.write_bytes((value & 0xFFFFFFFF).to_bytes(4, "little"))
    return xdma_write(addr, path)


def mmio_read_u32(addr: int, label: str = "") -> tuple[int, CmdResult]:
    name = label or f"rd_{addr:08X}"
    path = REG_DIR / f"{name}.bin"
    result = xdma_read(addr, 4, path)
    value = int.from_bytes(path.read_bytes()[:4], "little") if result.ok and path.exists() else 0
    return value, result


def compare_bytes(expected: bytes, actual_path: Path, max_diff: int = 16) -> tuple[bool, str]:
    if not actual_path.exists():
        return False, f"missing readback file: {actual_path}"
    actual = actual_path.read_bytes()
    if expected == actual:
        digest = hashlib.sha256(actual).hexdigest()
        return True, f"match: {len(actual)} bytes, sha256={digest}"
    lines = [f"mismatch: expected_len={len(expected)}, actual_len={len(actual)}"]
    for index, (a, b) in enumerate(zip(expected, actual)):
        if a != b:
            lines.append(f"0x{index:08X}: expected=0x{a:02X}, actual=0x{b:02X}")
            if len(lines) > max_diff:
                break
    return False, "\n".join(lines)


def assert_window_range(base: int, offset: int, size: int) -> None:
    assert base in (GLOBAL_BASE, IBUF_BASE, OBUF_BASE)
    assert offset >= 0 and size > 0
    assert offset + size <= WINDOW_BYTES, f"range exceeds 64KB window: off=0x{offset:X}, size=0x{size:X}"


_PREFLIGHT_DONE = False
_PREFLIGHT_OK = False
_PREFLIGHT_DETAIL = {}


def preflight_global_bram(force: bool = False) -> dict:
    """Check that XDMA can reach global_bram before running CDMA/PE traffic."""
    global _PREFLIGHT_DONE, _PREFLIGHT_OK, _PREFLIGHT_DETAIL
    if _PREFLIGHT_DONE and not force:
        return {"pass": _PREFLIGHT_OK, "cached": True, **_PREFLIGHT_DETAIL}

    payload = bytes(((0x5A + i) & 0xFF) for i in range(BRAM_WORD_BYTES))
    write_path = DATA_DIR / "preflight_global_write.bin"
    read_path = READBACK_DIR / "preflight_global_read.bin"
    write_path.write_bytes(payload)

    wr = xdma_write(GLOBAL_BASE, write_path, timeout=10)
    print_result("PREFLIGHT XDMA WRITE global_bram", wr)
    if not wr.ok:
        _PREFLIGHT_DONE = True
        _PREFLIGHT_OK = False
        _PREFLIGHT_DETAIL = {"stage": "write_global_bram", "returncode": wr.returncode, "stderr": wr.stderr}
        return {"pass": False, **_PREFLIGHT_DETAIL}

    rd = xdma_read(GLOBAL_BASE, len(payload), read_path, timeout=10)
    print_result("PREFLIGHT XDMA READ global_bram", rd)
    if not rd.ok:
        _PREFLIGHT_DONE = True
        _PREFLIGHT_OK = False
        _PREFLIGHT_DETAIL = {"stage": "read_global_bram", "returncode": rd.returncode, "stderr": rd.stderr}
        return {"pass": False, **_PREFLIGHT_DETAIL}

    match, detail = compare_bytes(payload, read_path)
    _PREFLIGHT_DONE = True
    _PREFLIGHT_OK = match
    _PREFLIGHT_DETAIL = {"stage": "compare_global_bram", "detail": detail}
    print("PREFLIGHT COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    return {"pass": match, **_PREFLIGHT_DETAIL}


def run_xdma_info(timeout: int = 10) -> CmdResult | None:
    if XDMA_INFO_EXE is None:
        print("xdma_info.exe not found")
        return None
    result = run_cmd([str(XDMA_INFO_EXE)], timeout=timeout)
    print_result("XDMA INFO", result)
    return result


def probe_read_u32_abs(addr: int, label: str, timeout: int = 5) -> dict:
    path = READBACK_DIR / f"probe_{label}_rd32.bin"
    result = xdma_read(addr, 4, path, timeout=timeout)
    print_result(f"PROBE READ {label} @ 0x{addr:08X}", result)
    value = int.from_bytes(path.read_bytes()[:4], "little") if result.ok and path.exists() else None
    if value is not None:
        print(f"VALUE {label}: 0x{value:08X}")
    return {"label": label, "op": "read32", "addr": addr, "ok": result.ok, "returncode": result.returncode, "value": value, "stderr": result.stderr}


def probe_rw_window(base: int, label: str, timeout: int = 5) -> dict:
    payload = bytes(((0x30 + i) & 0xFF) for i in range(BRAM_WORD_BYTES))
    write_path = DATA_DIR / f"probe_{label}_wr.bin"
    read_path = READBACK_DIR / f"probe_{label}_rd.bin"
    write_path.write_bytes(payload)

    wr = xdma_write(base, write_path, timeout=timeout)
    print_result(f"PROBE WRITE {label} @ 0x{base:08X}", wr)
    if not wr.ok:
        return {"label": label, "op": "write", "addr": base, "ok": False, "returncode": wr.returncode, "stderr": wr.stderr}

    rd = xdma_read(base, len(payload), read_path, timeout=timeout)
    print_result(f"PROBE READ {label} @ 0x{base:08X}", rd)
    if not rd.ok:
        return {"label": label, "op": "read", "addr": base, "ok": False, "returncode": rd.returncode, "stderr": rd.stderr}

    match, detail = compare_bytes(payload, read_path)
    print(f"PROBE COMPARE {label}:", "PASS" if match else "FAIL")
    print(detail)
    return {"label": label, "op": "rw_compare", "addr": base, "ok": match, "detail": detail}


def diagnose_xdma_address_map(run_info: bool = True) -> list[dict]:
    """Probe every XDMA-visible slave before running the DCIM test."""
    if run_info:
        run_xdma_info()

    results = []
    # Register reads are least invasive. If these also timeout, the XDMA M_AXI path is not responding at all.
    for addr, label in [
        (CDMA_BASE + CDMA_SR, "cdma_sr"),
        (PE_STATUS_BASE + 0x00, "pe_status"),
        (PE_STATUS_BASE + 0x08, "pe_error_code"),
    ]:
        results.append(probe_read_u32_abs(addr, label))

    # BRAM windows should support write/readback without starting CDMA or PE.
    for base, label in [
        (GLOBAL_BASE, "global_bram"),
        (IBUF_BASE, "pe_ibuf"),
        (OBUF_BASE, "pe_obuf"),
    ]:
        results.append(probe_rw_window(base, label))

    print("\nSUMMARY")
    for item in results:
        status = "PASS" if item.get("ok") else "FAIL"
        print(f"{status:4s} {item['label']:14s} {item['op']:10s} @ 0x{item['addr']:08X}")
    return results

In [3]:
def cdma_status() -> int:
    value, result = mmio_read_u32(CDMA_BASE + CDMA_SR, "cdma_sr")
    assert result.ok, "CDMA status read failed"
    return value


def cdma_wait_idle(timeout_sec: float = 2.0) -> int:
    deadline = time.time() + timeout_sec
    last = 0
    while time.time() < deadline:
        last = cdma_status()
        if last & CDMA_SR_ERR_MASK:
            raise RuntimeError(f"CDMA error status: 0x{last:08X}")
        if last & CDMA_SR_IDLE:
            return last
        time.sleep(0.001)
    raise TimeoutError(f"CDMA did not become idle, last status=0x{last:08X}")


def cdma_reset() -> None:
    result = mmio_write_u32(CDMA_BASE + CDMA_CR, 1 << 2, "cdma_reset")
    print_result("CDMA RESET", result)
    assert result.ok
    time.sleep(0.01)
    cdma_wait_idle(timeout_sec=2.0)


def cdma_memcpy(src_addr: int, dst_addr: int, size: int, label: str = "cdma") -> None:
    assert size > 0
    assert src_addr % 4 == 0 and dst_addr % 4 == 0
    cdma_wait_idle(timeout_sec=2.0)
    for reg_name, reg_off, value in [
        ("SA", CDMA_SA, src_addr),
        ("DA", CDMA_DA, dst_addr),
        ("BTT", CDMA_BTT, size),
    ]:
        result = mmio_write_u32(CDMA_BASE + reg_off, value, f"{label}_{reg_name.lower()}")
        print_result(f"CDMA WRITE {reg_name}", result)
        assert result.ok, f"CDMA write {reg_name} failed"
    status = cdma_wait_idle(timeout_sec=5.0)
    print(f"CDMA {label} complete, status=0x{status:08X}")


In [4]:
def pack_activation_rows(act: np.ndarray) -> bytes:
    assert act.dtype == np.int8 and act.ndim == 2 and act.shape[1] == 16
    payload = bytearray()
    for row in range(0, act.shape[0], 2):
        lo = act[row].astype(np.uint8).tobytes()
        hi = act[row + 1].astype(np.uint8).tobytes() if row + 1 < act.shape[0] else bytes(16)
        payload += lo + hi
    return bytes(payload)


def pack_weight_tile(weight: np.ndarray) -> bytes:
    assert weight.dtype == np.int8 and weight.shape == (16, 8)
    return weight.astype(np.uint8).tobytes(order="C")


def golden_int8_matmul(act: np.ndarray, weight: np.ndarray, acc: int = 0) -> np.ndarray:
    assert act.dtype == np.int8 and weight.dtype == np.int8
    out = act.astype(np.int32) @ weight.astype(np.int32)
    if acc in (0, 1):
        return out.astype(np.int32)
    assert acc in (2, 4, 8, 16), "RTL accepts only acc=0/1/2/4/8/16"
    assert out.shape[0] % acc == 0, "raw_rows must be divisible by acc"
    return out.reshape(out.shape[0] // acc, acc, out.shape[1]).sum(axis=1).astype(np.int32)


def pack_expected_rows(rows: np.ndarray) -> bytes:
    assert rows.dtype == np.int32 and rows.ndim == 2 and rows.shape[1] == 8
    return rows.astype("<i4", copy=False).tobytes(order="C")


def decode_result_rows(payload: bytes) -> np.ndarray:
    assert len(payload) % 32 == 0
    return np.frombuffer(payload, dtype="<i4").reshape(-1, 8)


def make_case(raw_rows: int = 16, acc: int = 0, seed: int = 1234) -> dict:
    assert raw_rows > 0
    rng = np.random.default_rng(seed)
    act = rng.integers(-128, 128, size=(raw_rows, 16), dtype=np.int8)
    weight = rng.integers(-128, 128, size=(16, 8), dtype=np.int8)
    return make_case_from_arrays(act, weight, acc=acc)


def make_case_from_arrays(act: np.ndarray, weight: np.ndarray, acc: int = 0) -> dict:
    assert act.dtype == np.int8 and act.ndim == 2 and act.shape[1] == 16
    assert weight.dtype == np.int8 and weight.shape == (16, 8)
    expected = golden_int8_matmul(act, weight, acc=acc)
    return {
        "raw_rows": act.shape[0],
        "acc": acc,
        "act": act,
        "weight": weight,
        "weight_payload": pack_weight_tile(weight),
        "act_payload": pack_activation_rows(act),
        "expected_rows": expected,
        "expected_payload": pack_expected_rows(expected),
    }


def make_signed_extreme_case(raw_rows: int = 16, acc: int = 2) -> dict:
    assert raw_rows > 0
    values = np.array([-128, -127, -64, -3, -1, 0, 1, 2, 3, 7, 15, 31, 63, 64, 126, 127], dtype=np.int8)
    act = np.vstack([np.roll(values, row) for row in range(raw_rows)]).astype(np.int8)
    weight = np.vstack([np.roll(values[:8], col) for col in range(16)]).astype(np.int8)
    return make_case_from_arrays(act, weight, acc=acc)


def summarize_case(case: dict, name: str) -> None:
    print(
        f"{name}: rows={case['raw_rows']}, acc={case['acc']}, "
        f"expected_rows={case['expected_rows'].shape[0]}, "
        f"payloads={len(case['weight_payload'])}/{len(case['act_payload'])}/{len(case['expected_payload'])} bytes"
    )


case = make_case(raw_rows=16, acc=0, seed=0xDC10)
print("act shape     :", case["act"].shape)
print("weight shape  :", case["weight"].shape)
print("expected shape:", case["expected_rows"].shape)
print("payload bytes :", len(case["weight_payload"]), len(case["act_payload"]), len(case["expected_payload"]))


act shape     : (16, 16)
weight shape  : (16, 8)
expected shape: (16, 8)
payload bytes : 128 256 512


In [5]:
def pe_status() -> dict:
    status, status_result = mmio_read_u32(PE_STATUS_BASE + 0x00, "pe_status")
    error_code, err_result = mmio_read_u32(PE_STATUS_BASE + 0x08, "pe_error_code")
    assert status_result.ok and err_result.ok, "PE status read failed"
    return {
        "raw": status,
        "busy": bool(status & 0x1),
        "done": bool(status & 0x2),
        "error": bool(status & 0x4),
        "config_ready": bool(status & 0x8),
        "error_code": error_code,
    }


def pe_ctrl_value(acc: int, start: bool = False, config_valid: bool = True) -> int:
    return ((1 if start else 0) << 0) | ((1 if config_valid else 0) << 1) | (MODE_INT8 << 2) | ((acc & 0x1F) << 5)


def pe_clear_done_if_needed() -> None:
    status = pe_status()
    if status["done"] and not status["busy"]:
        result = mmio_write_u32(PE_CTRL_BASE, 0x1, "pe_clear_done")
        print_result("PE CLEAR DONE", result)
        assert result.ok
        time.sleep(0.002)
        result = mmio_write_u32(PE_CTRL_BASE, 0x0, "pe_clear_done_low")
        assert result.ok


def pe_start(wsrc_base: int, asrc_base: int, dst_base: int, raw_rows: int, acc: int) -> dict:
    assert wsrc_base % BRAM_WORD_BYTES == 0
    assert asrc_base % BRAM_WORD_BYTES == 0
    assert dst_base % BRAM_WORD_BYTES == 0
    pe_clear_done_if_needed()
    for name, base, value in [
        ("wsrc", PE_CFG0_BASE, wsrc_base),
        ("asrc", PE_CFG1_BASE, asrc_base),
        ("dst", PE_CFG2_BASE, dst_base),
        ("raw_rows", PE_CFG3_BASE, raw_rows),
    ]:
        result = mmio_write_u32(base, value, f"pe_cfg_{name}")
        print_result(f"PE CFG {name}", result)
        assert result.ok

    ctrl = pe_ctrl_value(acc, start=False, config_valid=True)
    result = mmio_write_u32(PE_CTRL_BASE, ctrl, "pe_ctrl_cfg")
    print_result("PE CTRL config_valid", result)
    assert result.ok
    time.sleep(0.002)

    result = mmio_write_u32(PE_CTRL_BASE, pe_ctrl_value(acc, start=True, config_valid=True), "pe_ctrl_start")
    print_result("PE CTRL start", result)
    assert result.ok
    time.sleep(0.002)
    result = mmio_write_u32(PE_CTRL_BASE, ctrl, "pe_ctrl_start_low")
    assert result.ok

    deadline = time.time() + 5.0
    last = {}
    while time.time() < deadline:
        last = pe_status()
        if last["error"]:
            raise RuntimeError(f"PE error: {last}")
        if last["done"]:
            return last
        time.sleep(0.001)
    raise TimeoutError(f"PE did not complete, last status={last}")


## XDMA address-map diagnosis
Run this before the DCIM case if preflight times out. If every probe times out, debug XDMA enumeration/reset/link first. If only one window fails, debug that BD slave/address segment.


In [6]:
# Optional: run this when preflight_global_bram() fails.
# diag = diagnose_xdma_address_map()
# diag


In [7]:
def run_dcim_case(case: dict, name: str = "case0") -> dict:
    w_global_off = 0x0000
    a_global_off = 0x1000
    r_global_off = 0x4000
    w_ibuf_off = 0x0000
    a_ibuf_off = 0x0100
    r_obuf_off = 0x0000

    weight_payload = case["weight_payload"]
    act_payload = case["act_payload"]
    expected_payload = case["expected_payload"]

    for base, off, size in [
        (GLOBAL_BASE, w_global_off, len(weight_payload)),
        (GLOBAL_BASE, a_global_off, len(act_payload)),
        (GLOBAL_BASE, r_global_off, len(expected_payload)),
        (IBUF_BASE, w_ibuf_off, len(weight_payload)),
        (IBUF_BASE, a_ibuf_off, len(act_payload)),
        (OBUF_BASE, r_obuf_off, len(expected_payload)),
    ]:
        assert_window_range(base, off, size)

    preflight = preflight_global_bram()
    assert preflight["pass"], (
        "XDMA/global_bram preflight failed before DCIM. "
        "Check that the xdma_dcim_bram bitstream is programmed, XDMA is enumerated, "
        f"and global_bram is mapped at 0x{GLOBAL_BASE:08X}. Detail: {preflight}"
    )

    weight_path = DATA_DIR / f"{name}_weight.bin"
    act_path = DATA_DIR / f"{name}_act.bin"
    out_path = READBACK_DIR / f"{name}_result.bin"
    weight_path.write_bytes(weight_payload)
    act_path.write_bytes(act_payload)

    wr_w = xdma_write(GLOBAL_BASE + w_global_off, weight_path)
    print_result("XDMA WRITE global weight", wr_w)
    assert wr_w.ok
    wr_a = xdma_write(GLOBAL_BASE + a_global_off, act_path)
    print_result("XDMA WRITE global activation", wr_a)
    assert wr_a.ok

    cdma_reset()
    cdma_memcpy(GLOBAL_BASE + w_global_off, IBUF_BASE + w_ibuf_off, len(weight_payload), f"{name}_w_to_ibuf")
    cdma_memcpy(GLOBAL_BASE + a_global_off, IBUF_BASE + a_ibuf_off, len(act_payload), f"{name}_a_to_ibuf")

    status = pe_start(
        wsrc_base=w_ibuf_off,
        asrc_base=a_ibuf_off,
        dst_base=r_obuf_off,
        raw_rows=case["raw_rows"],
        acc=case["acc"],
    )
    print("PE DONE:", status)

    cdma_memcpy(OBUF_BASE + r_obuf_off, GLOBAL_BASE + r_global_off, len(expected_payload), f"{name}_obuf_to_global")
    rd = xdma_read(GLOBAL_BASE + r_global_off, len(expected_payload), out_path)
    print_result("XDMA READ global result", rd)
    assert rd.ok

    match, detail = compare_bytes(expected_payload, out_path)
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    if out_path.exists():
        print("hardware rows:")
        print(decode_result_rows(out_path.read_bytes()))
        print("expected rows:")
        print(case["expected_rows"])
    assert match, "DCIM result mismatch"
    return {"pass": match, "status": status, "readback": out_path}


result = run_dcim_case(case, name=f"int8_rows{case['raw_rows']}_acc{case['acc']}")
result

PREFLIGHT XDMA WRITE global_bram: PASS, ret=0, elapsed=0.119s
CMD: /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/bin/xdma_rw.exe h2c_0 write 0x10000000 -b -f /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/xdma_dcim_bram/data/preflight_global_write.bin
STDOUT:
32 bytes read from file /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/xdma_dcim_bram/data/preflight_global_write.bin
32 bytes written in 0.000052s
PREFLIGHT XDMA READ global_bram: PASS, ret=0, elapsed=0.089s
CMD: /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/bin/xdma_rw.exe c2h_0 read 0x10000000 -l 0x20 -b -f /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/xdma_dcim_bram/readbacks/preflight_global_read.bin
STDOUT:
32 bytes received in 0.000049s
PREFLIGHT COMPARE: PASS
match: 32 bytes, sha256=787146bf61595710bb14fc328b554cfa42b23191287b5e45f890426f97838600
XDMA WRITE global weight: PASS, ret=0, elapsed=0.089s
CMD: /home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/bin/xdma_rw.exe 

{'pass': True,
 'status': {'raw': 10,
  'busy': False,
  'done': True,
  'error': False,
  'config_ready': True,
  'error_code': 0},
 'readback': PosixPath('/home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/xdma_dcim_bram/readbacks/int8_rows16_acc0_result.bin')}

In [8]:
# Additional regression cases. Run this cell after the smoke test above.
regression_cases = [
    ("int8_rows16_acc1_random", make_case(raw_rows=16, acc=1, seed=0xDC11)),
    ("int8_rows16_acc4_random", make_case(raw_rows=16, acc=4, seed=0xDC14)),
    ("int8_rows15_acc0_odd_rows", make_case(raw_rows=15, acc=0, seed=0xDC15)),
    ("int8_rows16_acc2_extremes", make_signed_extreme_case(raw_rows=16, acc=2)),
    ("int8_rows32_acc8_random", make_case(raw_rows=32, acc=8, seed=0xDC18)),
]

for name, test_case in regression_cases:
    summarize_case(test_case, name)

# regression_results = [run_dcim_case(test_case, name=name) for name, test_case in regression_cases]
# regression_results
